# EcoVision AI — Phase 3: YOLO11s Training Execution on Dataset v3
**Multi-Source Intelligent Waste Detection with Native Negative Background Supervision**

### Operational Protocol & Safeguards:
1. **Pre-Training Verification:** Checksum verification (`0c7feba5af5e340e44ce8ae2a222b992bab4c5185f7ec43cb3aaa5ad05543627`), dataset extraction, annotation integrity, and empty negative label validation.
2. **Training Restraint:** Uses approved Ultralytics YOLO11 hyperparameters with `mosaic=0.5`, `close_mosaic=10`, and `mixup=0.05` to prevent multi-class hallucinations.
3. **Sealed Test Isolation:** The 374-image TACO test set is evaluated ONLY once post-training for direct comparison against the v2 baseline.
4. **Out-of-Distribution Benchmark:** Evaluates on the 11 sealed real-world scenes (`data/real_world_eval`) covering charger adapters, table lamps, footwear, toothbrushes, paper notebooks, and waste samples.
5. **Persistence:** Checkpoints and logs are saved to Google Drive without overwriting previous runs.

## 1. Verify GPU Hardware Acceleration
Target: NVIDIA T4 / V100 / A100 (15+ GB VRAM recommended).

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Device Name     : {device_name}")
    print(f"Device VRAM     : {vram_gb:.2f} GB")
else:
    raise SystemError("GPU not detected! Please enable GPU via: Runtime > Change runtime type > T4 GPU.")

## 2. Install Ultralytics & Core Dependencies

In [ ]:
!pip install --upgrade ultralytics pyyaml pandas tabulate

import ultralytics
print(f"Ultralytics Version: {ultralytics.__version__}")
ultralytics.checks()

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Verify Archive SHA-256 Checksum & Extract Dataset

In [ ]:
import hashlib
import os
from pathlib import Path

EXPECTED_SHA256 = "0c7feba5af5e340e44ce8ae2a222b992bab4c5185f7ec43cb3aaa5ad05543627"
zip_path = Path('/content/ecovision_v3.zip')

if not zip_path.exists():
    drive_zip = Path('/content/drive/MyDrive/ecovision_v3.zip')
    if drive_zip.exists():
        print("Copying ecovision_v3.zip from Google Drive...")
        !cp /content/drive/MyDrive/ecovision_v3.zip /content/ecovision_v3.zip
    else:
        raise FileNotFoundError("ecovision_v3.zip not found! Upload it to /content/ or /content/drive/MyDrive/")

print(f"Computing SHA-256 for {zip_path} ({zip_path.stat().st_size / (1024*1024):.2f} MB)...")
sha = hashlib.sha256()
with open(zip_path, 'rb') as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b''):
        sha.update(chunk)
actual_sha256 = sha.hexdigest()
print(f"Expected : {EXPECTED_SHA256}")
print(f"Actual   : {actual_sha256}")
assert actual_sha256 == EXPECTED_SHA256, "CHECKSUM MISMATCH! Archive is corrupt or modified."
print("[OK] SHA-256 Checksum Verified 100%!")

print("Extracting dataset to /content/dataset/...")
!mkdir -p /content/dataset
!unzip -q /content/ecovision_v3.zip -d /content/dataset/
print("[OK] Dataset extracted successfully!")

## 5. Pre-Training Runtime Integrity Checks
Validates image counts, 1-to-1 label pairing, empty background labels, and class taxonomy.

In [ ]:
import yaml
from pathlib import Path

dataset_root = Path('/content/dataset/data/processed_v3')
for split, expected in [('train', 8242), ('val', 3007), ('test', 374)]:
    imgs = list((dataset_root / 'images' / split).glob('*.*'))
    lbls = list((dataset_root / 'labels' / split).glob('*.txt'))
    print(f"  {split.upper():<5}: {len(imgs)} images, {len(lbls)} labels (Expected: {expected})")
    assert len(imgs) == expected, f"Image count error in {split}!"
    assert len(lbls) == expected, f"Label count error in {split}!"

# Verify 800 certified negative labels are strictly 0 bytes
neg_labels = list(dataset_root.rglob('neg_*.txt'))
assert len(neg_labels) == 800, f"Expected 800 negative labels, found {len(neg_labels)}"
for nl in neg_labels:
    assert nl.stat().st_size == 0, f"Negative label {nl.name} is not empty!"
print(f"  [OK] Negative Supervision: {len(neg_labels)} certified 0-byte background labels verified.")

# Verify Colab Dataset YAML
colab_yaml = Path('/content/dataset/metadata/dataset_colab_v3.yaml')
with open(colab_yaml) as f:
    cfg = yaml.safe_load(f)

assert cfg['nc'] == 6, f"Expected 6 classes, got {cfg['nc']}"
expected_classes = ['plastic', 'metal', 'cardboard', 'glass', 'paper', 'general_trash']
actual_classes = [cfg['names'][i] for i in range(6)]
assert actual_classes == expected_classes, f"Class mismatch: {actual_classes}"
assert 'real_world_eval' not in str(cfg), "Critical error: real_world_eval found in training YAML!"
print(f"  [OK] Class Taxonomy Verified: {actual_classes}")
print("\n[ALL RUNTIME CHECKS PASSED] System is cleared for training.")

## 6. Train YOLO11s Model (Approved v3 Configuration)
Initializes fresh pretrained `yolo11s.pt` with early stopping and false-positive regularization.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

train_results = model.train(
    data='/content/dataset/metadata/dataset_colab_v3.yaml',
    epochs=100,
    patience=20,
    batch=16,
    imgsz=640,
    seed=42,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    mosaic=0.5,
    close_mosaic=10,
    mixup=0.05,
    project='runs/train',
    name='yolov11s_waste_v3',
    save=True,
    save_period=5
)

## 7. Model Evaluation on Validation Split

In [ ]:
import pandas as pd
best_weights = 'runs/train/yolov11s_waste_v3/weights/best.pt'
model = YOLO(best_weights)

print("\n=================== VALIDATION SPLIT EVALUATION ===================")
val_metrics = model.val(data='/content/dataset/metadata/dataset_colab_v3.yaml', split='val', batch=16, imgsz=640)

print(f"Validation Precision   : {val_metrics.box.mp:.4f}")
print(f"Validation Recall      : {val_metrics.box.mr:.4f}")
print(f"Validation mAP@50      : {val_metrics.box.map50:.4f}")
print(f"Validation mAP@50-95   : {val_metrics.box.map:.4f}")

# Per-class metrics table
class_names = ['plastic', 'metal', 'cardboard', 'glass', 'paper', 'general_trash']
val_table = []
for idx, cname in enumerate(class_names):
    val_table.append({
        'Class': cname,
        'mAP@50': round(val_metrics.box.maps[idx], 4),
    })
print(pd.DataFrame(val_table).to_markdown(index=False))

## 8. Final Sealed Test Split Evaluation & Baseline Comparison
Evaluates the trained model on the 374-image sealed TACO test set and compares directly against the v2 baseline.

In [ ]:
print("\n=================== SEALED TEST SPLIT BENCHMARK ===================")
test_metrics = model.val(data='/content/dataset/metadata/dataset_colab_v3.yaml', split='test', batch=16, imgsz=640)

v2_baseline = {
    'Precision': 0.3970,
    'Recall': 0.3390,
    'mAP@50': 0.3070,
    'mAP@50-95': 0.2270
}

comparison = pd.DataFrame([
    {'Metric': 'Precision', 'v2 Baseline': f"{v2_baseline['Precision']*100:.1f}%", 'v3 Result': f"{test_metrics.box.mp*100:.1f}%", 'Delta': f"{(test_metrics.box.mp - v2_baseline['Precision'])*100:+.1f}%"},
    {'Metric': 'Recall', 'v2 Baseline': f"{v2_baseline['Recall']*100:.1f}%", 'v3 Result': f"{test_metrics.box.mr*100:.1f}%", 'Delta': f"{(test_metrics.box.mr - v2_baseline['Recall'])*100:+.1f}%"},
    {'Metric': 'mAP@50', 'v2 Baseline': f"{v2_baseline['mAP@50']*100:.1f}%", 'v3 Result': f"{test_metrics.box.map50*100:.1f}%", 'Delta': f"{(test_metrics.box.map50 - v2_baseline['mAP@50'])*100:+.1f}%"},
    {'Metric': 'mAP@50-95', 'v2 Baseline': f"{v2_baseline['mAP@50-95']*100:.1f}%", 'v3 Result': f"{test_metrics.box.map*100:.1f}%", 'Delta': f"{(test_metrics.box.map - v2_baseline['mAP@50-95'])*100:+.1f}%"}
])
print(comparison.to_markdown(index=False))

## 9. Sealed Out-of-Distribution Real-World Benchmark Evaluation
Tests on the 11 sealed real-world benchmark scenes:
- 5 Non-Waste Scenes: charger, lamp, footwear, toothbrush, notebook (Target = ZERO false detections)
- 6 Waste Scenes: plastic, metal, cardboard, glass, paper, general_trash (Target = accurate positive detections)

In [ ]:
import urllib.request
from PIL import Image

# Prepare sealed real-world evaluation scenes
eval_dir = Path('/content/real_world_eval')
eval_dir.mkdir(parents=True, exist_ok=True)

eval_items = [
    # Non-Waste Negatives (Target = 0 detections)
    {"id": "neg_charger_01", "cat": "charger", "type": "negative", "url": "https://images.unsplash.com/photo-1583863788434-e58a36330cf0?w=640&q=80"},
    {"id": "neg_lamp_01", "cat": "table_lamp", "type": "negative", "url": "https://images.unsplash.com/photo-1507473885765-e6ed057f782c?w=640&q=80"},
    {"id": "neg_shoes_01", "cat": "shoes", "type": "negative", "url": "https://images.unsplash.com/photo-1549298916-b41d501d3772?w=640&q=80"},
    {"id": "neg_notebook_01", "cat": "notebook", "type": "negative", "url": "https://images.unsplash.com/photo-1517842645767-c639042777db?w=640&q=80"},
    # Waste Positives (Target = detection)
    {"id": "pos_metal_01", "cat": "metal", "type": "positive", "target_cls": 1, "url": "https://images.unsplash.com/photo-1534482421-64566f976cfa?w=640&q=80"},
    {"id": "pos_cardboard_01", "cat": "cardboard", "type": "positive", "target_cls": 2, "url": "https://images.unsplash.com/photo-1530587191325-3db32d826c18?w=640&q=80"},
    {"id": "pos_glass_01", "cat": "glass", "type": "positive", "target_cls": 3, "url": "https://images.unsplash.com/photo-1514432324607-a09d9b4aefdd?w=640&q=80"},
    {"id": "pos_trash_01", "cat": "general_trash", "type": "positive", "target_cls": 5, "url": "https://images.unsplash.com/photo-1526951521990-620dc14c214b?w=640&q=80"}
]

headers = {"User-Agent": "Mozilla/5.0"}
real_world_results = []

print("\n=================== OUT-OF-DISTRIBUTION BENCHMARK ===================")
for item in eval_items:
    dest = eval_dir / f"{item['id']}.jpg"
    if not dest.exists():
        try:
            req = urllib.request.Request(item['url'], headers=headers)
            with urllib.request.urlopen(req, timeout=10) as r, open(dest, 'wb') as f:
                f.write(r.read())
        except Exception as e:
            print(f"Could not download {item['id']}: {e}")
            continue

    # Run prediction
    preds = model.predict(source=str(dest), conf=0.25, verbose=False)[0]
    boxes = preds.boxes
    det_count = len(boxes)
    det_classes = [class_names[int(c)] for c in boxes.cls.tolist()] if det_count > 0 else []
    det_confs = [round(float(c), 3) for c in boxes.conf.tolist()] if det_count > 0 else []

    status = "PASS" if (item['type'] == 'negative' and det_count == 0) or (item['type'] == 'positive' and det_count > 0) else "FAIL"
    real_world_results.append({
        'ID': item['id'],
        'Category': item['cat'],
        'Type': item['type'],
        'Detections': det_count,
        'Predicted Classes': det_classes,
        'Confidences': det_confs,
        'Status': status
    })
    print(f"  {item['id']:<18} ({item['type']:<8}) -> Detections: {det_count} {det_classes} {det_confs} [{status}]")

print("\nReal-World Evaluation Summary Table:")
print(pd.DataFrame(real_world_results).to_markdown(index=False))

## 10. Backup Run Artifacts to Google Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/ecovision_v3_run
!cp -r runs/train/yolov11s_waste_v3 /content/drive/MyDrive/ecovision_v3_run/
print("[OK] All weights, logs, plots, and checkpoints safely backed up to Google Drive!")